In [2]:
from sqlalchemy import create_engine
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

def create_connection():

    load_dotenv()
    host = os.environ.get('DB_DESTINATION_HOST')
    port = os.environ.get('DB_DESTINATION_PORT')
    db = os.environ.get('DB_DESTINATION_NAME')
    username = os.environ.get('DB_DESTINATION_USER')
    password = os.environ.get('DB_DESTINATION_PASSWORD')
    
    conn = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{db}')
    return conn

# установите соединение с базой
conn = create_connection()

In [20]:
data=pd.read_sql(f"SELECT * FROM flat_target_price", conn)
print(data)

            id  floor  is_apartment  kitchen_area  living_area  rooms  studio  \
0            0      9         False          9.90    19.900000      1   False   
1            1      7         False          0.00    16.600000      1   False   
2            2      9         False          9.00    32.000000      2   False   
3            3      1         False         10.10    43.099998      3   False   
4            4      3         False          3.00    14.000000      1   False   
...        ...    ...           ...           ...          ...    ...     ...   
141357  141357     16         False         11.00    18.000000      1   False   
141358  141358      5         False          5.28    28.330000      2   False   
141359  141359      7         False          5.30    20.000000      1   False   
141360  141360     15         False         13.80    33.700001      2   False   
141361  141361     16         False          7.60    18.000000      1   False   

        total_area     pric

In [17]:
columns_name=data.columns.tolist()
print(columns_name)

['id', 'floor', 'is_apartment', 'kitchen_area', 'living_area', 'rooms', 'studio', 'total_area', 'price', 'build_year', 'building_type_int', 'latitude', 'longitude', 'ceiling_height', 'flats_count', 'floors_total', 'has_elevator']


In [18]:
num_cols = ['total_area','price','flats_count']
print(data[num_cols])

        total_area     price  flats_count
0        35.099998   9500000           84
1        43.000000  13500000           97
2        56.000000  13500000           80
3        76.000000  20000000          771
4        24.000000   5200000          208
...            ...       ...          ...
141357   42.000000  10500000          672
141358   41.110001   7400000           80
141359   31.500000   9700000           72
141360   65.300003  11750000          480
141361   38.000000   8000000          128

[141362 rows x 3 columns]


In [21]:
threshold = 1.5
potential_outliers = pd.DataFrame()

for col in num_cols:
	Q1 = data[col].quantile(0.25)# Ваш код здесь #
	Q3 = data[col].quantile(0.75)# Ваш код здесь #
	IQR = Q3 - Q1# Ваш код здесь #
	margin = threshold*IQR # Ваш код здесь #
	lower = Q1 - margin # Ваш Код здесь #
	upper = Q3 + margin # Ваш Код здесь #
	potential_outliers[col] = ~data[col].between(lower, upper)

outliers = potential_outliers.any(axis=1)
data=data[~outliers].reset_index(drop=True)

print(data)

            id  floor  is_apartment  kitchen_area  living_area  rooms  studio  \
0            0      9         False          9.90    19.900000      1   False   
1            1      7         False          0.00    16.600000      1   False   
2            2      9         False          9.00    32.000000      2   False   
3            4      3         False          3.00    14.000000      1   False   
4            5      9         False          0.00     0.000000      2   False   
...        ...    ...           ...           ...          ...    ...     ...   
120911  141356      8         False          6.00    42.000000      3   False   
120912  141358      5         False          5.28    28.330000      2   False   
120913  141359      7         False          5.30    20.000000      1   False   
120914  141360     15         False         13.80    33.700001      2   False   
120915  141361     16         False          7.60    18.000000      1   False   

        total_area     pric